## Итераторы и итерируемое

Начнем с базы. Внутри Python есть две вещи: iterator и iterable. В чем разница и что это такое?

Iterable - это объект, над которым можно проводить итерацию. Что такое итерация? По сути это процесс перебора элементов (например, строки/множества/списки - это итерируемые объекты)

А что же тогда такое итератор? А это у нас объект, который занимается процессом итерации. По определению это класс, у которого реализованы методы next и iter (для iterable объекта реализуется только сам iter)


In [3]:
c = [1, 2, 3, 4]
for i in c: # что здесь происходит? Неявно вызывается iter(c)
# Причем iter() работает для так называемых контейнеров (для всех, у кого есть __getitem__ или __iter__)
    print(i)

1
2
3
4


In [4]:
iter(c)

In [5]:
for c in 32: #поэтому по числам не получится, они не итерируемые
    print(c)

TypeError: 'int' object is not iterable

In [6]:
n = iter(c)
print(next(n))
print(next(n))
print(next(n))
print(next(n))
print(next(n))

1
2
3
4


StopIteration: 

In [7]:
l = [1, 2, 3]
next(l) #список iterable, но не итератор

TypeError: 'list' object is not an iterator

In [16]:
d = {1: "1", (1,2,3): "2", 1.0: "3"}

iter_d = iter(d)
print(d[1.0])
print(next(iter_d))
print(next(iter_d))
print(next(iter_d)) # не работает перемещение по итератору с float числами

3
1
(1, 2, 3)


StopIteration: 

In [21]:
d = {1: "1", (1,2,3): "2", "1.0": "3"}

iter_d = iter(d)
print(d[1.0]) # не явно. Получение по float ключу значения из "аналогичного" int ключа
print(next(iter_d))
print(next(iter_d))
print(next(iter_d))
# print(next(iter_d))

for i in d:
    print(i)

1
1
(1, 2, 3)
1.0
1
(1, 2, 3)
1.0


Где можно встретить итераторы? Да на самом деле много где!

Например, функция zip возвращает итератор:

In [23]:
a = [1, 2, 3]
b = [1, 2, 3]
c = zip(a, b)
print(type(c))
print(next(c))
print(next(c))
print(next(c))

<class 'zip'>
(1, 1)
(2, 2)
(3, 3)


А также есть функция enumerate() - она делает нумерацию элементов, что можно впоследствие использовать внутри for:

In [28]:
k = [4, 5, 6]
k_e = enumerate(k)
print(*next(k_e))
print(*next(k_e))
print(*next(k_e))

0 4
1 5
2 6


In [25]:
for num, el in enumerate(k):
    print(num, el)

0 4
1 5
2 6


### А как сделать свой собственный итератор?

Что есть итератор? С точки зрения Python - это объект, который обладает dunder методом \_\_next\_\_, который берет и возвращает следующий элемент до самого конца. В тот момент, когда происходит конец, он должен возвращать ошибку StopIteration

Давайте делать простой итератор на строках:

In [43]:
class MyIter:
    def __init__(self, s):
        self.s = s
        self.index = 0

    def __next__(self):
        if self.index >= len(self.s):
            raise StopIteration("Ended")
        else:
            val = self.index, self.s[self.index]
            self.index += 1
            return val

a = "abcdefg"
a_iter = MyIter(a)
# a_iter = iter(a)
print(next(a_iter))
print(next(a_iter))
print(next(a_iter))
print(next(a_iter))
print(next(a_iter))
print(next(a_iter))
print(next(a_iter))

(0, 'a')
(1, 'b')
(2, 'c')
(3, 'd')
(4, 'e')
(5, 'f')
(6, 'g')


In [41]:
print(next(a_iter))
print(next(a_iter))

StopIteration: Ended

Ура, сделали простой рабочий итератор! Как вы можете заметить, внутри next можно делать что угодно (можно изменять элементы, выводить по условиям и так далее). Но этого недостаточно, потому что давайте попробуем сделать следующее:

In [44]:
a = "abcdefg"
my_iter = MyIter(a)
for i in my_iter:
    print(i)

TypeError: 'MyIter' object is not iterable

Опа, вроде как сделали итератор, но он не iterable. Как так?

Чтобы итератор стал полным итератором, нам надо добавить dunder метод \_\_iter\_\_, который и должен возвращать объект (вернее то, что нужно итерировать):

In [19]:
class MyIter:
    def __init__(self, s):
        self.s = s
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.s):
            raise StopIteration("Ended")
        else:
            self.index += 1
            return self.s[self.index - 1]

In [20]:
a = "abcdefg"
my_iter = MyIter(a)
for i in my_iter:
    print(i)

a
b
c
d
e
f
g


Зачем нужны итераторы?

* Позволяет перебирать элементы (без итераторов нам бы пришлось делать while, обращаться к каждому элементу). По факту проще синтаксис, не более

* Позволяет экономить память


Давайте попробуем сделать свой собственный range:

In [51]:
class MyRange:
    def __init__(self, *args):
        self.start = start
        self.end = end
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.start >= self.end:
            raise StopIteration
        else:
            value = self.start
            self.start += self.step
            return value

for i in range(3):
    print(i)
print()
for i in MyRange(3):
    print(i)
print()

for i in range(1, 20, 2):
    print(i)

print()
for i in MyRange(start=1, end=20, step=2):
    print(i)

print()
for i in range(start=1, end=20, step=2):
    print(i)

0
1
2


1
3
5
7
9
11
13
15
17
19

1
3
5
7
9
11
13
15
17
19



TypeError: range() takes no keyword arguments

## Генераторы

В чем разница между генератором и итератором?

* Итератор - объект, позволяющий пройтись по объектам

* Генератор - объект, позволяющий генерировать значения

Формально генератор - это расширение итератора (за счет того, что можно генерировать данные)

Как распознать генератор? Простым словом yield!

In [52]:
def simple_gen():
    yield "Такса"
    yield "Корги"
    yield "Лабрадор"
    yield "Доберман"
    yield "Английский мастиф"

s = simple_gen()
print(next(s))
print(next(s))
print(next(s))
print(next(s))
print(next(s))
print(next(s)) # опа, знакомая нам ошибка


Такса
Корги
Лабрадор
Доберман
Английский мастиф


StopIteration: 

In [53]:
for i in simple_gen():
    print(i)

Такса
Корги
Лабрадор
Доберман
Английский мастиф


Как работает генератор, что за yield?

Все просто:

Мы вызываем функцию. Когда он доходит до yield, то выдает значение. После этого функция переходит как бы в режим ожидания. Когда мы ее вызываем в следующий раз, он начинает с места, где закончил и продолжает. Как закончить жизнь генератора? Сделать return

In [60]:
def fibonacci(n):
    print("Start")
    a, b, counter = 0, 1, 0
    while True:
        if (counter > n):
            return
        yield a
        (a, b) = (b, a + b)
        counter += 1

f = fibonacci(3)
print(f)
print("Before loop")
# for x in f:
#     print("Start loop")
#     print(x)
print("Start loop")
print(next(f))
print("Start loop")
print(next(f))
print("Start loop")
print(next(f))
print("Start loop")
print(next(f))
print("Start loop")
print(next(f))

<generator object fibonacci at 0x108b29e00>
Before loop
Start loop
Start
0
Start loop
1
Start loop
1
Start loop
2
Start loop


StopIteration: 

Ну хорошо, в нашем первом примере как-то это явно неудобно, писать кучу строк... А можно, на самом деле, сделать вот такую штуку:

In [61]:
def simple_gen():
    yield from ["Такса", "Корги", "Лабрадор", "Доберман", "Английский мастиф"] #yield from работает с iterable объектами

s = simple_gen()
print(next(s))
print(next(s))
print(next(s))
print(next(s))
print(next(s))

Такса
Корги
Лабрадор
Доберман
Английский мастиф


In [65]:
def yield_from_generator(n):
    a = yield from fibonacci(n)
    print(a)

yfg = yield_from_generator(5)
for i in yfg:
    print(i, end=" ")

Start
0 1 1 2 3 5 None


Где вы могли видеть генераторы? А вот тут:

In [32]:
(i for i in range(10))

<generator object <genexpr> at 0x10696f1c0>

Что еще крутого есть в генераторах? Для итераторов мы можем только ходить по значениям и делать с ними что-то (итерироваться). А вот в генераторы мы можем отправлять значения!

В чем прикол: на самом деле yield не только дает значения, но и послыает значения. Если воспринимать генератор как итератор, то разницы никакой, в общем-то, а вот если расширять функции генератора, то не совсем.

По дефолту yield выдает None, а поскольку мы ничего с этим не делаем, то как бы и ок. Но мы можем отправить что-то с помощью send(), и таким образом, модернизировать значения!

In [64]:
def count(firstval=0, step=1):
    counter = firstval
    while True:
        new_counter_val = yield counter # здесь возвращается либо None, либо результат send
        if new_counter_val is None:
            counter += step
        else:
            counter = new_counter_val[0]
            step = new_counter_val[1]

        if counter == -1:
            return

start_value = 2.1
step_value = 0.3
counter = count(start_value, step_value)
for i in range(10):
    new_value = next(counter)
    print(f"{new_value:2.2f}", end=", ")

print()
print("set current count value to another value:")
new_val = counter.send((100.5, 2)) # ооотправляем посылочку
# print(f"{new_val:2.2f}", end=", ")
for i in range(10):
    new_value = next(counter)
    print(f"{new_value:2.2f}", end=", ")

counter.send((-1,0))

next(counter)

2.10, 2.40, 2.70, 3.00, 3.30, 3.60, 3.90, 4.20, 4.50, 4.80, 
set current count value to another value:
102.50, 104.50, 106.50, 108.50, 110.50, 112.50, 114.50, 116.50, 118.50, 120.50, 

StopIteration: 

А еще можем вкидывать ошибки)))

In [66]:
def count(firstval=0, step=1):
    counter = firstval
    while True:
        try:
            new_counter_val = yield counter
            print("Iteration")
            if new_counter_val is None:
                counter += step
            else:
                counter = new_counter_val
        except Exception:
            yield (firstval, step, counter)

c = count()
for i in range(6):
    print(next(c))
print("Our state")
state_of_count = c.throw(Exception)
print(state_of_count)
for i in range(3):
    print(next(c))
c.close()
next(c)

0
Iteration
1
Iteration
2
Iteration
3
Iteration
4
Iteration
5
Our state
(0, 1, 5)
5
Iteration
6
Iteration
7


StopIteration: 

И на всякий случай, с чего начали, к тому и пришли: генераторы тоже можно передавать в качестве аргументов функции, опа, доп обертка

In [42]:
def firstn(generator, n):
    g = generator()
    for i in range(n):
        yield next(g)

print(list(firstn(simple_gen, 3)))

['Такса', 'Корги', 'Лабрадор']
